In [9]:
#De Duplicate
import pandas as pd
df = pd.read_csv("../data/satd_comment.csv")
seen = set()
indexes_to_drop = []
for index, row in df.iterrows():
    if row['text'] in seen:
        indexes_to_drop.append(index)
    seen.add(row['text'])
df.drop(indexes_to_drop, inplace=True)
df.to_csv("../data/satd_comment_2.csv", index=False)

In [13]:
#Splitting
import pandas as pd
df = pd.read_csv('../data/satd_comment.csv')
df = df.sample(frac=1, random_state=42)
df.iloc[:200].to_csv('../data/train.csv', index=False)
df.iloc[200:].to_csv('../data/test.csv', index=False)


In [15]:
#Add Not SATD into Train
import pandas as pd
def remove_duplicate(duplicated_df):
    seen = set()
    indexes_to_drop = []
    for index, row in duplicated_df.iterrows():
        if row['text'] in seen:
            indexes_to_drop.append(index)
        seen.add(row['text'])
    duplicated_df.drop(indexes_to_drop, inplace=True)
    return duplicated_df

df = pd.read_csv('../data/comment.csv')
df = df.sample(frac=1, random_state=42)
df = remove_duplicate(df)

train_df = pd.concat([pd.read_csv('../data/train.csv'), df.iloc[:200]], ignore_index=True)
train_df = train_df.sample(frac=1, random_state=42)
train_df = remove_duplicate(train_df)
train_df.to_csv('../data/train.csv', index=False)


In [52]:
#Add Not SATD into test
import pandas as pd
def remove_duplicate(duplicated_df):
    seen = set()
    indexes_to_drop = []
    for index, row in duplicated_df.iterrows():
        if row['text'] in seen:
            indexes_to_drop.append(index)
        seen.add(row['text'])
    duplicated_df.drop(indexes_to_drop, inplace=True)
    return duplicated_df

df = pd.read_csv('../data/comment.csv')
df = df.sample(frac=1, random_state=42)
df = remove_duplicate(df)

test_df = pd.read_csv('../data/test.csv')
seen = set(test_df['text'])
for x in set(pd.read_csv('../data/train.csv')['text']):
    seen.add(x)
rows = []
for index, row in df.iterrows():
    if row['text'] not in seen:
        rows.append(row)
    seen.add(row['text'])
    if len(rows) >= 85:
        break
test_df = pd.concat([test_df, pd.DataFrame(rows)], ignore_index=True)
test_df = test_df.sample(frac=1, random_state=42)
test_df.to_csv('../data/test.csv', index=False)


In [56]:
#Splitting
import pandas as pd
test_df = pd.read_csv('../data/test.csv')
train_df = pd.read_csv('../data/train.csv')
train_set = set(train_df['text'])
test_set = set(test_df)
intersection = set(pd.merge(train_df, test_df, on='text')['text'])
# intersection = set(train_df['text'])
drop_indexes = []
for index, row in test_df.iterrows():
    if row['text'] in intersection:
        drop_indexes.append(index)
# test_df.drop([0,1,3], inplace=True)
test_df.drop(drop_indexes, inplace=True)
# test_df.to_csv('../data/test.csv', index=False)
print(drop_indexes)


[]


In [20]:
#Add Not SATD into Train
import pandas as pd

train_temp = df[ df['id'].isin(set(pd.read_csv('../data/train.csv')['id'])) ]
test_temp = df[ df['id'].isin(set(pd.read_csv('../data/test.csv')['id'])) ]


train_temp.to_csv('../data/train_temp.csv')
test_temp.to_csv('../data/test_temp.csv')

print(len(train_temp[train_temp['satd'] == 'yes']))
print(len(test_temp[test_temp['satd'] == 'yes']))

wo_df = df[(df['repository'] != 69) & (df['satd'] == 'yes')]

all_len = len(df)
print(all_len)
satd_len = len(df[df['satd'] == 'yes'])
print(satd_len)
print(all_len//satd_len)
df = pd.read_csv('../data/comment.csv')
open_api_df = df[(df['repository'] == 69) & (df['satd'] == 'yes')]

print(len(open_api_df))
print(len(set(open_api_df[open_api_df['satd'] == 'yes']['comment'])))
print(len(set(df[df['satd'] == 'yes']['comment'])))




180
73
26229
589
44
229
130
450


In [20]:
#Split into train and test dataset
import pandas as pd
from sklearn.utils import shuffle

df = pd.read_csv('../data/comment.csv')
#Filter out comments containing  non ASCII characters
df = df[df['comment'].str.match('^[\x00-\x7F]*$', na=False)]
df = df[df['repository'] != 69]

df = shuffle(df)
df = df.sample(frac=1, random_state=43).reset_index(drop=True)
print('--------------Detect-------------')
print(f'Total {len(df[df["satd"] == "yes"])}/{len(df)}')

split_index = int(len(df) * 0.8)
df1 = df.iloc[:split_index]
df2 = df.iloc[split_index:]

df1 = df1.drop(columns=['type'])
df1.rename(columns={'comment': 'text', 'satd': 'label'}, inplace=True)

df2 = df2.drop(columns=['type'])
df2.rename(columns={'comment': 'text', 'satd': 'label'}, inplace=True)

df1.to_csv('../data/detect_train.csv', index=False)
df2.to_csv('../data/detect_test.csv', index=False)
print(f'train {len(df1[df1["label"] == "yes"])}/{len(df1)}')
print(f'test {len(df2[df2["label"] == "yes"])}/{len(df2)}')


df = df[df['satd'] == 'yes']
df = shuffle(df)
df = df.sample(frac=1, random_state=43).reset_index(drop=True)
print(f'-------------Classification {len(df)} - {len(df["type"].unique())}-------------')
for t in df['type'].unique():
    count = len(df[df['type'] == t])
    print(f"{t}: {count}/{len(df)}")


split_index = int(len(df) * 0.8)
df1 = df.iloc[:split_index]
df2 = df.iloc[split_index:]

df1 = df1.drop(columns=['satd'])
df1.rename(columns={'comment': 'text', 'type': 'label'}, inplace=True)

df2 = df2.drop(columns=['satd'])
df2.rename(columns={'comment': 'text', 'type': 'label'}, inplace=True)

df1.to_csv('../data/classify_train.csv', index=False)
df2.to_csv('../data/classify_test.csv', index=False)
print(f'----------Train  {len(df1)}/{len(df)} - {len(df1["label"].unique())}/{len(df["type"].unique())} -------------')
for t in df1['label'].unique():
    count = len(df1[df1['label'] == t])
    print(f"{t}: {count}/{len(df1)}")

print(f'----------Test  {len(df2)}/{len(df)} - {len(df2["label"].unique())}/{len(df["type"].unique())} -------------')
for t in df2['label'].unique():
    count = len(df2[df2['label'] == t])
    print(f"{t}: {count}/{len(df2)}")
df = df.drop(columns=['satd', 'code_before','code_after', 'code_method'])
df.rename(columns={'comment': 'text', 'type': 'label'}, inplace=True)
df.to_csv('../data/satd_comment.csv', index=False)

--------------Detect-------------
Total 592/47994
train 483/38395
test 109/9599
-------------Classification 592 - 15-------------
defect: 68/592
requirement: 221/592
temporary-fix: 51/592
dependency: 13/592
how-to: 64/592
refactor: 18/592
documentation: 17/592
build: 5/592
superficial-test: 17/592
code: 15/592
impractical-case: 10/592
skip-test: 23/592
multi: 63/592
subset-test: 2/592
design: 5/592
----------Train  473/592 - 15/15 -------------
defect: 58/473
requirement: 169/473
temporary-fix: 44/473
dependency: 11/473
how-to: 50/473
refactor: 14/473
documentation: 12/473
build: 4/473
superficial-test: 15/473
code: 12/473
impractical-case: 8/473
skip-test: 17/473
multi: 53/473
subset-test: 1/473
design: 5/473
----------Test  119/592 - 14/15 -------------
requirement: 52/119
temporary-fix: 7/119
superficial-test: 2/119
multi: 10/119
how-to: 14/119
defect: 10/119
documentation: 5/119
build: 1/119
dependency: 2/119
code: 3/119
skip-test: 6/119
refactor: 4/119
impractical-case: 2/119
subs

In [1]:
import pandas as pd

df = pd.read_csv('../data/comment.csv')
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')
data_frames = [train_df, test_df]
for index, row in df.iterrows():
    id = row['id']
    code_before = row['code_before']
    code_after = row['code_after']
    for frame in data_frames:
        frame.loc[frame['id'] == id, 'code_before'] = code_before
        frame.loc[frame['id'] == id, 'code_after'] = code_after

train_df.to_csv('../data/train.csv', index=False)
test_df.to_csv('../data/test.csv', index=False)

In [19]:
import pandas as pd
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split




df = pd.read_csv('../data/comment.csv')
train_df = pd.read_csv('../data/detect_train.csv')
test_df = pd.read_csv('../data/detect_test.csv')
test_df = shuffle(test_df)
test_df = test_df.sample(frac=1, random_state=43).reset_index(drop=True)
df1, df2 = train_test_split(test_df, test_size=0.6, random_state=42)

print(len(test_df) + len(train_df))
print(len(test_df))
print(len(df1))
print(len(df2))
print(len(df2)/len(df)*100)
# split_index = len(test_df) // 2
# df1 = df.iloc[:split_index]  # First 50%
# df2 = df.iloc[split_index:]
# df1.to_csv('../data/train.csv', index=False)
# df2.to_csv('../data/test.csv', index=False)

print(len(df1[df1['label'] == 'yes']))
print(len(df2[df2['label'] == 'yes']))

print(len(train_df[train_df['label'] == 'yes']))
print(len(test_df[test_df['label'] == 'yes']))



# data_frames = [train_df, test_df]
# for index, row in df.iterrows():
#     id = row['id']
#     code_before = row['code_before']
#     code_after = row['code_after']
#     for frame in data_frames:
#         frame.loc[frame['id'] == id, 'code_before'] = code_before
#         frame.loc[frame['id'] == id, 'code_after'] = code_after
#
# train_df.to_csv('../data/train.csv', index=False)
# test_df.to_csv('../data/test.csv', index=False)

25894
12947
5178
7769
29.4280303030303
110
182
292
292


In [1]:
import pandas as pd
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
train_df = pd.read_csv('../data/detect_train.csv')
train_df2 = pd.read_csv('../data/train.csv')
pd.concat([train_df, train_df2], axis=0, ignore_index=True).to_csv('../data/detect_train.csv', index=False)

In [10]:
#Logistic Regression
from imblearn.under_sampling import RandomUnderSampler
from sklearn.utils import shuffle
under_sampler = RandomUnderSampler(sampling_strategy='auto', random_state=42)
lr_df_all = pd.read_csv('../data/detect_train.csv')
lr_df_all = shuffle(lr_df_all)
indices, _ = under_sampler.fit_resample(lr_df_all.index.values.reshape(-1, 1), lr_df_all['label'])
lr_df_resampled = lr_df_all.loc[indices.flatten()]
lr_df_resampled.reset_index(drop=True)
lr_df_resampled = shuffle(lr_df_resampled)
lr_df_resampled.to_csv('../data/detect_train_balanced.csv')
# lr_dataset = Dataset.from_pandas(lr_df_resampled)

In [ ]:
import pandas as pd
df = pd.read_csv('../data/detect_n_shot.csv')
ndf = df.copy()
ndf = ndf.drop(columns=['type'])
ndf.rename(columns={'comment': 'text', 'satd': 'label'}, inplace=True)
ndf.to_csv('../data/detect_n_shot.csv', index=False)

In [ ]:

import pandas as pd
import  ast
import numpy as np
from util import sha1
f = './cache/gemini-embedding-exp-03-07.csv'
df = pd.read_csv(f)
# df['hash'] = df.apply(lambda row: sha1(row['text']), axis=1)
# hash = df.pop('hash')
# df.insert(1, 'hash', hash)
# df = pd.read_csv('./cache/gemini-embedding-exp-03-07.csv')
print(len(df))
# df.head(10)

# encoding_map = {}
# print(df.iloc[595])
# incorrect_indexes = []
# for row_index, r in df.iterrows():
#     try:
#         encoding_map[r['hash']] = np.array(ast.literal_eval(r['embedding']), dtype=np.float32)
#     except:
#         incorrect_indexes.append(row_index)
#         print(row_index)
#         print(r)

# df.drop(incorrect_indexes, inplace=True)
# df.to_csv(f, index=False)

In [1]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, AutoTokenizer, AutoModelForSeq2SeqLM

# tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-base')
# model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base')

tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')
model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')


input_text = "translate English to French: How are you?"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids
outputs = model.generate(input_ids)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/home/shahidul/dev/project/academic/technical-debt/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Comment tu êtes-vous?


In [ ]:
# df = pd.read_csv('../data/maldonado_corrected.csv')
# df['label'] = df['satd_orig'].apply(lambda x: 'yes' if x == 1 else 'no')
# df['text'] = df['comment_text']
# df = df[["text", "label"]]
#
# under_sampler = RandomUnderSampler(sampling_strategy=1, random_state=42)
# X_resampled, y_resampled = under_sampler.fit_resample(df[['text']], df['label'])
#
# # Create balanced DataFrame
# df = pd.DataFrame({'text': X_resampled['text'], 'label': y_resampled})
# dataset = Dataset.from_pandas(df).train_test_split(test_size=0.03, seed=42)
# dataset = dataset.remove_columns(['__index_level_0__'])
# dataset

# Dataset Preparation project-wise 80-20 percent

In [11]:
#Split into train and test dataset
import pandas as pd
import numpy as np
from sklearn.utils import shuffle

df = pd.read_csv('../data/comment.csv')
#Filter out comments containing  non ASCII characters
df = df[df['comment'].str.match('^[\x00-\x7F]*$', na=False)]
df = df[df['repository'] != 69]
df = shuffle(df)
# df = df.sample(frac=1, random_state=43).reset_index(drop=True)
duplicated_df = df.copy(deep= True)
unique_df = df.copy(deep= True)
unique_df['comment_lower'] = unique_df['comment'].str.strip().str.lower()
unique_df = unique_df.drop_duplicates(subset="comment_lower", keep="first").reset_index(drop=True)
unique_df.drop(columns = ['comment_lower'], inplace = True)

project_ids = np.random.permutation(duplicated_df["repository"].dropna().astype(int).unique())
for (dataset_prefix, df) in [['duplicate', duplicated_df], ['unique', unique_df]]:
    total_nx = len(df[df['satd'] == 'yes'])
    total_ny = len(df[df['satd'] == 'no'])
    test_x = 0
    test_y = 0

    df_train = pd.DataFrame(columns=df.columns)
    df_test = pd.DataFrame(columns=df.columns)
    train_project_ids = set()
    test_project_ids = set()
    for pid in project_ids:
        px_df = df[(df['repository'] == pid) & (df['satd'] == 'yes')]
        py_df = df[(df['repository'] == pid) & (df['satd'] == 'no')]
        if len(df_test[df_test['satd'] == 'yes']) < 0.20 * total_nx:
            df_test = pd.concat([df_test, px_df])
            test_x += len(px_df)
            test_project_ids.add(pid)
        else:
            df_train = pd.concat([df_train, px_df])
            train_project_ids.add(pid)


        if len(df_test[df_test['satd'] == 'no']) < 0.20 * total_ny:
            df_test = pd.concat([df_test, py_df])
            test_y += len(py_df)
            test_project_ids.add(pid)
        else:
            df_train = pd.concat([df_train, py_df])
            train_project_ids.add(pid)
    df_train = shuffle(df_train)
    df_train.drop(columns=['type'], inplace=True)
    df_train.rename(columns={'comment': 'text', 'satd': 'label'}, inplace=True)
    df_train.to_csv(f'../data/{dataset_prefix}_detect_train.csv', index=False)

    df_test = shuffle(df_test)
    df_test.drop(columns=['type'], inplace=True)
    df_test.rename(columns={'comment': 'text', 'satd': 'label'}, inplace=True)
    df_test.to_csv(f'../data/{dataset_prefix}_detect_test.csv', index=False)


    print(f'-------------- {dataset_prefix} Detect-------------')
    print(f'common project ids {set(map(int, train_project_ids)) & set(map(int, test_project_ids))}')
    print(f'total {len(df[df["satd"] == "yes"])}/{len(df)} of {df["repository"].nunique()} projects')
    print(f'train {len(df_train[df_train["label"] == "yes"])}/{len(df_train)} of {df_train["repository"].nunique()} projects')
    print(f'test {len(df_test[df_test["label"] == "yes"])}/{len(df_test)} of {df_test["repository"].nunique()} projects')

    #
    #
    df = df[df['satd'] == 'yes']
    df = shuffle(df)
    # df = df.sample(frac=1, random_state=43).reset_index(drop=True)
    print(f'------------- {dataset_prefix} Classification {len(df)} - {len(df["type"].unique())}-------------')
    for t in df['type'].unique():
        count = len(df[df['type'] == t])
        print(f"{t}: {count}/{len(df)}")
    #
    #
    split_index = int(len(df) * 0.0)
    df1 = df.iloc[:split_index]
    df2 = df.iloc[split_index:]

    df1 = df1.drop(columns=['satd'])
    df1.rename(columns={'comment': 'text', 'type': 'label'}, inplace=True)

    df2 = df2.drop(columns=['satd'])
    df2.rename(columns={'comment': 'text', 'type': 'label'}, inplace=True)

    df1.to_csv(f'../data/{dataset_prefix}_classify_train.csv', index=False)
    df2.to_csv(f'../data/{dataset_prefix}_classify_test.csv', index=False)
    df = df.drop(columns=['satd', 'code_before','code_after', 'code_method'])
    df.rename(columns={'comment': 'text', 'type': 'label'}, inplace=True)
    df.to_csv(f'../data/{dataset_prefix}_satd_comment.csv', index=False)

-------------- duplicate Detect-------------
common project ids {537, 623, 285, 41}
total 615/47994 of 488 projects
train 478/38380 of 382 projects
test 137/9614 of 108 projects
------------- duplicate Classification 615 - 15-------------
defect: 69/615
requirement: 225/615
temporary-fix: 54/615
impractical-case: 10/615
how-to: 65/615
skip-test: 23/615
dependency: 16/615
multi: 65/615
documentation: 18/615
superficial-test: 17/615
code: 22/615
refactor: 19/615
subset-test: 2/615
design: 5/615
build: 5/615
-------------- unique Detect-------------
common project ids set()
total 516/32555 of 466 projects
train 403/26024 of 368 projects
test 113/6531 of 98 projects
------------- unique Classification 516 - 15-------------
multi: 63/516
code: 19/516
defect: 60/516
requirement: 155/516
how-to: 63/516
dependency: 15/516
superficial-test: 17/516
skip-test: 22/516
refactor: 18/516
impractical-case: 9/516
temporary-fix: 54/516
subset-test: 2/516
documentation: 10/516
build: 4/516
design: 5/516


In [8]:
import pandas as pd

DATASET_NAME = 'duplicate'

classify_train_df = pd.read_csv(f'../data/{DATASET_NAME}_classify_train.csv')
classify_test_df = pd.read_csv(f'../data/{DATASET_NAME}_classify_test.csv')

train_ids = set(classify_train_df['text'])
filtered_test_df = classify_test_df[~classify_test_df['text'].isin(train_ids)]

filtered_test_df.to_csv(f'../data/{DATASET_NAME}_classify_test.csv', index=False)


Add hash column

In [ ]:
import os
import pandas as pd
from util import *
for base_dir in ['../data', '../cache/output/merged', '../cache/output/mismatched', '../cache/output/snapshot']:
    files = list(filter(lambda file: file.endswith('.csv') and not file.endswith('repository.csv') , os.listdir(base_dir)))
    for file in map(lambda file: os.path.join(base_dir, file), files):
        df = pd.read_csv(file)
        print(file)
        if 'hash' not in df.columns:
            if 'text' in df.columns:
                target_column = 'text'
            elif 'comment' in df.columns:
                target_column = 'comment'
            else:
                target_column = None
            df.insert(df.columns.get_loc(target_column) + 1, 'hash', df[target_column].map(lambda x: sha1(x)))
            df.to_csv(file, index=False)
